# 🧹 La limpieza de datos

**Nadie habla sobre la limpieza de datos.**
¿Sabes por qué? Porque es aburrido. No se vuelve viral en LinkedIn. No puedes poner «experto en eliminar filas duplicadas» en tu currículum sin parecer idiota.

**Pero hay algo que todo el mundo olvida mencionar:**
Si no puedes limpiar los datos, no puedes hacer ciencia de datos.



## 🙈 La cruda realidad de los datos reales

**Kaggle nos está mintiendo :)**

¿Esos conjuntos de datos limpios, perfectamente formateados, con nombres de columnas bonitos y sin valores perdidos? Eso no es la vida real. Eso es el Disneylandia de los datos.

Los datos reales se ven así:


## ❓ ¿Por qué la limpieza de datos es realmente importante (más que tu modelo)?

Hay algo que no enseñan en los cursos de ML:

Un modelo decente con datos limpios es mejor que un modelo perfecto con datos basura. Siempre. Sin excepción.

Un científicos de datos puede pasar semanas ajustando hiperparámetros, probando diferentes arquitecturas, leyendo artículos sobre mecanismos de atención... con datos que tenían un 40 % de filas duplicadas y fechas del año 2087.

*¿La precisión de su modelo? 94 %.*

¿Su valor comercial real? Cero. **Porque «si entra basura, sale basura»** no es solo un dicho. Es la ley del universo.

La regla del 80/20 en la ciencia de datos es en realidad más bien del 80/15/5:
- 80 % de datos de limpieza
- 15 % ingeniería de características
- 5 % de modelado real

¿Ese último 5 %? Es la parte en la que todo el mundo se centra. ¿El otro 95 %? Ahí es donde se realiza el trabajo real.


## 🛠️ Las herramientas de Python
### 🐼 1. Pandas: el caballo de batalla

Lo básico que todo el mundo sabe:

In [ ]:
import pandas as pd

df = pd.read_csv('messy_data.csv')
df = df.dropna()  # Eliminar valores perdidos
df = df.drop_duplicates()  # Eliminar duplicados

Eso resuelve quizás el 5 % de los problemas reales.
Lo que realmente debemos hacer:


In [ ]:
# Leer con las opciones adecuadas, ya que los archivos nunca estan limpios.
df = pd.read_csv(
    'messy_data.csv',
    encoding='latin-1',  # Porque UTF-8 falla la mitad de las veces.
    sep=',', # Separador correcto del archivo
    decimal='.',  # Maneje números como "1234.56"
    thousands=',',  # Maneje números como "1,234.56"
    na_values=['NA', 'N/A', 'null', 'NULL', '', ' ', 'None'], # Valores que deben ser tratados como NaN
    parse_dates=['date_column'], # Columnas que deben ser parseadas como fechas
    date_format='mixed'  # Para pandas >= 2.0, reemplaza date_parser
)

# Eliminar los espacios en blanco de TODO, no solo de los extremos.
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
# Corregir los nombres de las columnas porque alguien utilizó espacios y caracteres especiales.
df.columns = df.columns.str.lower().str.replace(' ', '_', regex=False).str.replace('[^a-z0-9_]', '', regex=True)

### 🕳️ 2. El problema de los valores perdidos (que nunca es sencillo)

Los datos faltantes nunca son algo sencillo. 
No se trata simplemente de «eliminar las filas.

Enfoque incorrecto:


In [ ]:
df = df.dropna()  # Enhorabuena, acabas de eliminar el 80 % de tus datos.

Un poco mejor:

In [ ]:
# Vea dónde se encuentran realmente los datos que faltan.
print(df.isnull().sum())

# Quizás solo sea una columna la que está mal.
df = df.dropna(subset=['critical_column'])
# Rellenar las columnas numéricas con la mediana (no con la media, debido a los valores atípicos).
df['price'].fillna(df['price'].median(), inplace=True)
# Rellena la categoría con la moda o «Desconocido».
df['category'].fillna('Desconocido', inplace=True) # O bien:
df['category'].fillna(df['category'].mode()[0], inplace=True) # Rellena con la moda

Pero aquí está la verdadera pregunta que nadie se hace:¿Por qué faltan los datos?

A veces, los datos que faltan son aleatorios, otras veces se debe a que los clientes menores de 18 años no han facilitado datos sobre sus ingresos (porque no tienen ingresos). Eso no es aleatorio, es significativo.
Si simplemente eliminas esas filas, no estás limpiando los datos. Estás introduciendo un **sesgo**.


## 🔄 3. Cómo lidiar con los duplicados
Los duplicados no siempre son evidentes.


In [ ]:
# Esto encuentra duplicados exactos.
df.drop_duplicates()

# ¿Pero qué pasa con ESTOS?
# John Smith vs john smith vs John  Smith
# (555) 123-4567 vs 555-123-4567 vs 5551234567


La detección real de duplicados requiere una coincidencia aproximada:

In [ ]:
# Normalizar el texto antes de comprobarlo.
df['name_clean'] = df['name'].str.lower().str.strip()
df['phone_clean'] = df['phone'].str.replace('[^0-9]', '', regex=True)

# Ahora busca duplicados en las columnas limpias.
df = df.drop_duplicates(subset=['name_clean', 'phone_clean'])
# Luego elimine las columnas limpias si no las necesita.
df = df.drop(columns=['name_clean', 'phone_clean'])

## 🔢 4. Tipos de datos (porque todo es una cadena por defecto)
Python lee todo como cadenas de caracteres a menos que se le indique lo contrario. Esto lo estropea todo.


In [ ]:
# Esto parece estar bien, pero en realidad está roto.
df['price'] = '$49.99'
df['date'] = '2024-01-15'
df['quantity'] = '5'

# Las matemáticas no funcionan con cadenas de caracteres.
df['total'] = df['price'] * df['quantity']  # '$49.99' * '5' = ???


Arréglalo:

In [ ]:
# Eliminar los símbolos de moneda y convertir a flotante
df['price'] = df['price'].str.replace('[$,]', '', regex=True).astype(float)

# Convertir las fechas correctamente
df['date'] = pd.to_datetime(df['date'], errors='coerce')
# Convertir números
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')

El error='coerce' es clave. Convierte los valores erróneos en NaN en lugar de bloquearse. Entonces puedes decidir qué hacer con ellos.

## 📉 5. Valores atípicos (cuando tus datos tienen problemas de confianza)
Los valores atípicos no siempre son errores. A veces son fraudes. A veces son errores de introducción de datos. A veces son reales.


In [ ]:
# Enfoque estadístico: cualquier valor que supere las 3 desviaciones estándar.
mean = df['price'].mean()
std = df['price'].std()
df = df[(df['price'] < mean + 3*std) & (df['price'] > mean - 3*std)]

# Enfoque IQR (mejor para datos sesgados)
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
df = df[(df['price'] >= Q1 - 1.5*IQR) & (df['price'] <= Q3 + 1.5*IQR)]


Pero aquí está la cuestión: investiga siempre los valores atípicos antes de eliminarlos.

Esa transacción de 1 000 000 € podría ser un error tipográfico. O podría ser tu cliente más valioso. Si la eliminas automáticamente, podrías estar desechando una señal, no ruido


## 🧰 6. Las herramientas que me hubiera gustado conocer antes
**PyJanitor**— Hace que la limpieza de pandas sea menos dolorosa.


In [ ]:
# janitor en una librería útil para la limpieza de datos
# Si no la tenemos instalada: 
# %pip install pyjanitor

import janitor

df = (
    df
    .clean_names()  # Corregir automáticamente los nombres de las columnas
    .remove_empty()  # Eliminar filas/columnas vacías
    .coalesce(['col1', 'col2'], 'new_col')  # Combinar columnas
)


**Great Expectations** — Validación de datos que no es un rollo

In [ ]:
# Great Expectations para validación de datos y calidad de datos.
# Si no la tenemos instalada:
# %pip install great_expectations

import great_expectations as ge

# Great Expectations para validación de datos y calidad de datos.
df_ge = ge.from_pandas(df) # Convertir a un DataFrame de Great Expectations
# Definir expectativas
df_ge.expect_column_values_to_not_be_null('customer_id') # customer_id no debe tener valores nulos
df_ge.expect_column_values_to_be_between('age', 0, 120) # La edad debe estar entre 0 y 120
df_ge.expect_column_values_to_match_regex('email', r'^[\w\.-]+@[\w\.-]+\.\w+$') # Validar formato de email
# Validar el DataFrame
results = df_ge.validate()


Esto detecta los problemas ANTES de que rompan tu modelo.

**Pandas Profiling** — Comprenda sus datos en segundos

In [ ]:
# ydata-profiling es la versión actualizada de pandas_profiling
# Nos sirve para generar informes rápidos de calidad de datos.
# Si no la tenemos instalada:
# %pip install ydata-profiling

from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Informe de Calidad de Datos")
profile.to_file("report.html")

Genera un informe HTML completo con distribuciones, correlaciones, valores perdidos, todo. Ahorra horas de exploración manual.

## ⚙️ El flujo de trabajo de limpieza de datos que realmente funciona
Este es el proceso habitual. 
### 👁️ 1. Primer vistazo (5 minutos)


In [ ]:
# Forma y tipos
print(df.shape)
print(df.dtypes)

# Primeras y últimas filas (captura cosas extrañas al final)
print(df.head())
print(df.tail())

# Estadísticas resumidas
print(df.describe(include='all'))

# Valores perdidos
print(df.isnull().sum())


### 📝 2. Limpieza del nombre de la columna (2 minutos)


In [ ]:
# Hacer que las columnas sean más consistentes
df.columns = df.columns.str.lower().str.replace(' ', '_', regex=False)

### 🔧 3. Correcciones de tipos de datos (10 minutos)


In [ ]:
# Corrige los tipos ANTES de hacer cualquier otra cosa.
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['price'] = pd.to_numeric(df['price'].str.replace('[$,]', '', regex=True), errors='coerce')

### 🧩 4. Estrategia para valores perdidos (15 minutos)


In [ ]:
# Comprender POR QUÉ faltan datos
missing_pct = df.isnull().sum() / len(df) * 100
print(missing_pct[missing_pct > 0])

# Eliminar columnas con más del 50 % de datos faltantes (normalmente basura).
df = df.dropna(axis=1, thresh=len(df)*0.5)

# Gestionar el resto según la lógica empresarial.


### ✂️ 5. Eliminación de duplicados (10 minutos)


In [ ]:
# Buscar duplicados exactos
print(f"Duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates()

# Comprueba si hay duplicados difusos en las columnas clave# (Esto se personaliza en función de tus datos)


### 🕵️ 6. Investigación de valores atípicos (20 minutos)


In [ ]:
# Distribuciones de tramas
df.hist(figsize=(20,15))

# Comprueba los valores atípicos estadísticamente# Investiga, no te limites a eliminarlos


### ✅ 7. Validación (10 minutos)


In [ ]:
# Comprobaciones finales de cordura
# assert se usa para verificar condiciones que deben ser verdaderas
assert df['age'].min() >= 0, " Se encontraron edades negativas"
assert df['price'].min() >= 0, "Se encontraron precios negativos"
assert df.isnull().sum().sum() < len(df) * 0.1, "Demasiados valores nulos restantes"

Total: ~70 minutos para una primera pasada en datos de complejidad media.

### 🚫 Los errores que no debemos cometer:
#### **Error n.º 1**: Eliminar datos «malos» sin investigarlos

*Lección*: Investiga antes de borrar.

#### **Error n.º 2**: No conservar una copia de los datos originales.

*Lección*: Conserva siempre los datos originales. Limpia los datos en nuevas columnas/marcos de datos.

#### **Error n.º 3**: Dar por sentado que los formatos de datos son coherentes

*Lección*: Obtenga datos de muestra del principio, del medio y del final antes de escribir el código de limpieza.

#### **Error n.º 4**: No documentar las decisiones sobre limpieza.

*Lección*: Documenta TODO. Por qué eliminaste datos, por qué rellenaste valores nulos, todo.

### 🏁 Cuándo dejar de limpiar
No existen datos perfectos.
En algún momento, hay que decir «ya está bien» y seguir adelante. ¿Cómo saber cuándo?

Deténgase cuando:

1.	Las columnas críticas tienen menos del 5 % de datos faltantes.
2.	No quedan duplicados evidentes.
3.	Los tipos de datos son correctos.
4.	Se han investigado los valores atípicos (no necesariamente eliminados).
5.	Las comprobaciones básicas de integridad han superado la prueba.
6.	Puedes explicar tus decisiones de limpieza.

No te detengas porque:

1.	Estás aburrido.
2.	Está «tardando demasiado».
3.	Quieres empezar a modelar

El tiempo que dedicas ahora a limpiar te ahorra 10 veces más tiempo en depuración más adelante.
### 🤥 La incómoda verdad sobre la ciencia de datos
La mayoría de los científicos de datos dedican:
1.	El 80 % de su tiempo lo dedican a la limpieza de datos.
2.	15 % en ingeniería de características
3.	5 % sobre el modelado real

Pero en LinkedIn publican sobre:

1.	0 % de limpieza de datos
2.	20 % ingeniería de características
3.	80 % redes neuronales y modelos de vanguardia

¿Este desajuste? Es la razón por la que los científicos de datos junior tienen dificultades. Están formados para el 5 % y no preparados para el 80 %.

Si quieres ser valioso, sé bueno en las cosas poco atractivas. Cualquiera puede ejecutar model.fit(). No todo el mundo puede convertir datos basura en algo útil.


---

## 🏗️ Ejemplo Práctico Completo: Limpieza de Datos de Ventas

Vamos a aplicar todo lo aprendido en un ejemplo real con datos desordenados.

### 🎲 Paso 1: Crear datos desordenados de ejemplo


In [ ]:
import pandas as pd
import numpy as np
# El módulo StringIO nos permite leer datos desde una cadena como si fuera un archivo.
from io import StringIO

# Crear datos desordenados realistas
datos_sucios = """
CustomerID,Customer Name,Email,PURCHASE_DATE,Product,$$Price$$,Quantity,Status
1001,John Smith,john@email.com,2024-01-15,Laptop,$1,299.99,1,Completed
1002,JANE DOE,JANE@EMAIL.COM,15/01/2024,Mouse,$29.99,2,completed
1003,Bob  Johnson,bob@email,2024/01/16,Keyboard,"$49,99",1,Pending
1001,John Smith,john@email.com,2024-01-15,Laptop,$1299.99,1,Completed
,Anonymous User,,2024-01-17,Monitor,$199.99,NA,Completed
1004,Alice Brown,alice@email.com,2087-01-18,USB Cable,$9.99,100,Completed
1005,Charlie Wilson,charlie@email.com,2024-01-19,Headphones,NULL,1,Cancelled
1006,David Lee,david@email.com,01-20-24,Laptop,$1299.99,1,completed
1007,Emma Davis,emma@email.com,2024-01-21,Mouse,$29.99,-5,Completed
1008,Frank Miller,frank@email,2024-01-22,Keyboard,$49.99,2,Pending
1002,Jane Doe,jane@email.com,2024-01-15,Mouse,$29.99,2,Completed
"""

# Leer los datos
df_raw = pd.read_csv(StringIO(datos_sucios.strip()))

print("📊 Datos originales:")
print(f"Shape: {df_raw.shape}")
print("\n" + "="*80)
display(df_raw)

### 🩺 Paso 2: Diagnóstico inicial


In [ ]:
print("🔍 DIAGNÓSTICO DE PROBLEMAS:\n")

# 1. Tipos de datos
print("1️⃣ Tipos de datos:")
print(df_raw.dtypes)
print("\n" + "-"*80 + "\n")

# 2. Valores perdidos
print("2️⃣ Valores perdidos:")
print(df_raw.isnull().sum())
print(f"\nPorcentaje de valores perdidos:")
print((df_raw.isnull().sum() / len(df_raw) * 100).round(2))
print("\n" + "-"*80 + "\n")

# 3. Duplicados
print("3️⃣ Duplicados:")
print(f"Filas duplicadas completas: {df_raw.duplicated().sum()}")
print(f"Duplicados en CustomerID: {df_raw['CustomerID'].duplicated().sum()}")
print("\n" + "-"*80 + "\n")

# 4. Problemas en los datos
print("4️⃣ Problemas detectados:")
print(f"- Emails inválidos: {(~df_raw['Email'].str.contains('@', na=False) | ~df_raw['Email'].str.contains('.', na=False)).sum()}")
print(f"- Cantidades negativas: {(df_raw['Quantity'] < 0).sum()}")
print(f"- Precios con formato incorrecto: columna '$$Price$$' contiene '$' y ','")
print(f"- Nombres de columnas inconsistentes: espacios y caracteres especiales")
print(f"- Fechas futuras: {(pd.to_datetime(df_raw['PURCHASE_DATE'], errors='coerce') > pd.Timestamp.now()).sum()}")

### 🚿 Paso 3: Limpieza sistemática


In [ ]:
# Hacer una copia para limpiar (SIEMPRE preservar los datos originales)
df_clean = df_raw.copy()

print("🧹 PROCESO DE LIMPIEZA:\n")

# 1. Limpiar nombres de columnas
print("✅ 1. Limpiando nombres de columnas...")
df_clean.columns = (df_clean.columns
                    .str.lower()
                    .str.replace(' ', '_', regex=False)
                    .str.replace('[^a-z0-9_]', '', regex=True))
print(f"Columnas nuevas: {list(df_clean.columns)}")

# 2. Eliminar espacios en blanco
print("\n✅ 2. Eliminando espacios en blanco...")
df_clean = df_clean.map(lambda x: x.strip() if isinstance(x, str) else x)

# 3. Limpiar y convertir precio
print("\n✅ 3. Limpiando columna de precio...")
df_clean['price'] = (df_clean['price']
                     .str.replace('$', '', regex=False)
                     .str.replace(',', '', regex=False)
                     .str.replace('"', '', regex=False))
df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')
print(f"Tipo de dato de price: {df_clean['price'].dtype}")

# 4. Convertir fechas
print("\n✅ 4. Convirtiendo fechas...")
df_clean['purchase_date'] = pd.to_datetime(df_clean['purchase_date'], errors='coerce')
# Marcar fechas futuras como inválidas
df_clean.loc[df_clean['purchase_date'] > pd.Timestamp.now(), 'purchase_date'] = pd.NaT
print(f"Tipo de dato de purchase_date: {df_clean['purchase_date'].dtype}")

# 5. Limpiar quantity
print("\n✅ 5. Limpiando cantidades...")
df_clean['quantity'] = pd.to_numeric(df_clean['quantity'], errors='coerce')
# Eliminar cantidades negativas (datos erróneos)
df_clean.loc[df_clean['quantity'] < 0, 'quantity'] = np.nan

# 6. Normalizar customer_name
print("\n✅ 6. Normalizando nombres...")
df_clean['customer_name'] = df_clean['customer_name'].str.title()

# 7. Normalizar email
print("\n✅ 7. Normalizando emails...")
df_clean['email'] = df_clean['email'].str.lower()
# Marcar emails inválidos
email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
df_clean.loc[~df_clean['email'].str.match(email_pattern, na=False), 'email'] = np.nan

# 8. Normalizar status
print("\n✅ 8. Normalizando status...")
df_clean['status'] = df_clean['status'].str.capitalize()

print("\n" + "="*80)
print("Limpieza de formatos completada ✓")

### 👯 Paso 4: Manejo de duplicados


In [ ]:
print("🔄 MANEJO DE DUPLICADOS:\n")

# Ver duplicados antes de eliminar
print(f"Duplicados completos antes: {df_clean.duplicated().sum()}")
print("\nFilas duplicadas:")
display(df_clean[df_clean.duplicated(keep=False)].sort_values('customerid'))

# Eliminar duplicados exactos, manteniendo el primero
df_clean = df_clean.drop_duplicates(keep='first')

print(f"\n✅ Duplicados después: {df_clean.duplicated().sum()}")
print(f"Registros eliminados: {len(df_raw) - len(df_clean)}")
print(f"Registros restantes: {len(df_clean)}")

### 🔍 Paso 5: Manejo de valores perdidos


In [ ]:
print("🕳️ MANEJO DE VALORES PERDIDOS:\n")

# Análisis de valores perdidos
missing_analysis = pd.DataFrame({
    'Columna': df_clean.columns,
    'Valores_Perdidos': df_clean.isnull().sum().values,
    'Porcentaje': (df_clean.isnull().sum().values / len(df_clean) * 100).round(2)
})
missing_analysis = missing_analysis[missing_analysis['Valores_Perdidos'] > 0]

print("Análisis de valores perdidos:")
display(missing_analysis)

print("\n📋 Estrategia de manejo:\n")

# CustomerID: Eliminar filas sin ID (no podemos identificar al cliente)
print("1️⃣ CustomerID: Eliminando filas sin ID...")
filas_antes = len(df_clean)
df_clean = df_clean.dropna(subset=['customerid'])
print(f"   Filas eliminadas: {filas_antes - len(df_clean)}")

# Customer_name: Rellenar con 'Unknown'
print("\n2️⃣ Customer_name: Rellenando con 'Unknown'...")
df_clean['customer_name'].fillna('Unknown', inplace=True)

# Email: Rellenar con 'no-email@unknown.com'
print("\n3️⃣ Email: Rellenando con 'no-email@unknown.com'...")
df_clean['email'].fillna('no-email@unknown.com', inplace=True)

# Purchase_date: Eliminar filas (una compra sin fecha no es válida)
print("\n4️⃣ Purchase_date: Eliminando filas sin fecha...")
filas_antes = len(df_clean)
df_clean = df_clean.dropna(subset=['purchase_date'])
print(f"   Filas eliminadas: {filas_antes - len(df_clean)}")

# Price: Rellenar con la mediana (más robusta que la media)
print("\n5️⃣ Price: Rellenando con la mediana...")
median_price = df_clean['price'].median()
df_clean['price'].fillna(median_price, inplace=True)
print(f"   Mediana usada: ${median_price:.2f}")

# Quantity: Rellenar con 1 (asumimos 1 unidad si no se especifica)
print("\n6️⃣ Quantity: Rellenando con 1...")
df_clean['quantity'].fillna(1, inplace=True)

print("\n✅ Valores perdidos restantes:")
print(df_clean.isnull().sum())

### 🏁 Paso 6: Validación final


In [ ]:
print("✅ VALIDACIÓN FINAL:\n")

# Comprobaciones de integridad
validaciones_exitosas = 0
validaciones_totales = 7

# 1. No hay valores perdidos críticos
if df_clean['customerid'].isnull().sum() == 0:
    print("✓ 1. No hay CustomerIDs perdidos")
    validaciones_exitosas += 1
else:
    print("✗ 1. ERROR: Hay CustomerIDs perdidos")

# 2. No hay duplicados
if df_clean.duplicated().sum() == 0:
    print("✓ 2. No hay duplicados")
    validaciones_exitosas += 1
else:
    print("✗ 2. ERROR: Hay duplicados")

# 3. Precios son positivos
if (df_clean['price'] >= 0).all():
    print("✓ 3. Todos los precios son positivos")
    validaciones_exitosas += 1
else:
    print("✗ 3. ERROR: Hay precios negativos")

# 4. Cantidades son positivas
if (df_clean['quantity'] > 0).all():
    print("✓ 4. Todas las cantidades son positivas")
    validaciones_exitosas += 1
else:
    print("✗ 4. ERROR: Hay cantidades negativas o cero")

# 5. Fechas son válidas y no futuras
if (df_clean['purchase_date'] <= pd.Timestamp.now()).all():
    print("✓ 5. Todas las fechas son válidas")
    validaciones_exitosas += 1
else:
    print("✗ 5. ERROR: Hay fechas futuras")

# 6. Tipos de datos correctos
tipos_correctos = (
    df_clean['price'].dtype in ['float64', 'float32'] and
    df_clean['quantity'].dtype in ['float64', 'int64', 'float32', 'int32'] and
    df_clean['purchase_date'].dtype == 'datetime64[ns]'
)
if tipos_correctos:
    print("✓ 6. Tipos de datos correctos")
    validaciones_exitosas += 1
else:
    print("✗ 6. ERROR: Tipos de datos incorrectos")

# 7. Nombres de columnas limpios
if all(col.islower() and ' ' not in col for col in df_clean.columns):
    print("✓ 7. Nombres de columnas limpios")
    validaciones_exitosas += 1
else:
    print("✗ 7. ERROR: Nombres de columnas con problemas")

print(f"\n{'='*80}")
print(f"🎯 RESULTADO: {validaciones_exitosas}/{validaciones_totales} validaciones exitosas")
print(f"{'='*80}\n")

# Comparación antes/después
print("📊 COMPARACIÓN ANTES/DESPUÉS:\n")
comparison = pd.DataFrame({
    'Métrica': ['Filas totales', 'Columnas', 'Duplicados', 'Valores perdidos'],
    'Antes': [len(df_raw), len(df_raw.columns), df_raw.duplicated().sum(), df_raw.isnull().sum().sum()],
    'Después': [len(df_clean), len(df_clean.columns), df_clean.duplicated().sum(), df_clean.isnull().sum().sum()]
})
display(comparison)

print("\n📋 DATOS LIMPIOS FINALES:")
print(f"Shape: {df_clean.shape}")
print(f"Tipos: \n{df_clean.dtypes}")
print("\n")
display(df_clean)

---

## 📝 Ejercicios Prácticos

Ahora es tu turno de practicar. Aplica lo aprendido en estos ejercicios.

### 👥 Ejercicio 1: Limpieza de Datos de Empleados

Tienes un dataset de empleados con los siguientes problemas:
- Nombres con espacios extras
- Salarios con formato de moneda
- Fechas en diferentes formatos
- Emails inválidos
- Duplicados

**Tu tarea:** Limpia los datos siguiendo las mejores prácticas.

In [ ]:
# Datos de empleados con problemas
datos_empleados = """
EmployeeID,Full Name,Email Address,Hire Date,Salary,Department
E001,  John   Smith  ,JOHN.SMITH@COMPANY.COM,2020-01-15,"$75,000",Sales
E002,Jane Doe,jane.doe@company,15/02/2020,$85000,Marketing
E003,Bob Johnson,bob@company.com,2020/03/20,"$95,500",IT
E001,John Smith,john.smith@company.com,2020-01-15,$75000,Sales
E004,Alice  Brown,alice.brown@company.com,20-04-2020,$70000,HR
E005,Charlie Wilson,,2020-05-10,"$,120,000",IT
E006,David Lee,david@,2020-06-15,NULL,Marketing
E007,Emma Davis,emma.davis@company.com,2025-07-20,$65000,Sales
"""

# Cargar los datos
df_empleados = pd.read_csv(StringIO(datos_empleados.strip()))

# ESCRIBE TU CÓDIGO AQUÍ
# 1. Limpia los nombres de las columnas
# 2. Elimina espacios en blanco
# 3. Normaliza nombres
# 4. Limpia y convierte salarios
# 5. Convierte fechas
# 6. Valida emails
# 7. Elimina duplicados
# 8. Maneja valores perdidos
# 9. Valida los datos finales

# Tu código aquí...

### 📊 Ejercicio 2: Detección y Manejo de Valores Atípicos

Tienes datos de transacciones con posibles valores atípicos.

In [ ]:
# Generar datos con valores atípicos
np.random.seed(42)
transacciones_normales = np.random.normal(100, 20, 95)
# Añadir algunos valores atípicos
transacciones_outliers = np.array([500, 1000, -50, 10000, 750])
transacciones = np.concatenate([transacciones_normales, transacciones_outliers])

df_transacciones = pd.DataFrame({
    'transaction_id': range(1, len(transacciones) + 1),
    'amount': transacciones
})

# ESCRIBE TU CÓDIGO AQUÍ
# 1. Visualiza la distribución de los datos (histogram o boxplot)
# 2. Detecta valores atípicos usando el método IQR
# 3. Detecta valores atípicos usando desviación estándar (Z-score)
# 4. Investiga los valores atípicos (muéstralos)
# 5. Decide qué hacer con ellos (justifica tu decisión)

# Tu código aquí...

### 🔤 Ejercicio 3: Estandarización de Datos de Texto

Tienes datos de clientes con inconsistencias en el formato de texto.

In [ ]:
datos_clientes = """
customer_id,phone,country,postal_code
1,(555) 123-4567,USA,12345
2,555-234-5678,usa,12345-6789
3,5553456789,United States,12345
4,+1-555-456-7890,US,ABCDE
5,(555) 567 8901,U.S.A.,12345
"""

df_clientes = pd.read_csv(StringIO(datos_clientes.strip()))

# ESCRIBE TU CÓDIGO AQUÍ
# 1. Estandariza los números de teléfono a un formato único (por ejemplo: 5551234567)
# 2. Normaliza los nombres de países
# 3. Valida y limpia códigos postales
# 4. Crea una función reutilizable para la limpieza de teléfonos

# Tu código aquí...

### 🚇 Ejercicio 4: Pipeline Completo de Limpieza

Crea una función reutilizable que limpie cualquier dataset siguiendo buenas prácticas.

In [ ]:
def clean_dataframe(df, config=None):
    """
    Pipeline de limpieza de datos reutilizable.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame a limpiar
    config : dict, optional
        Configuración de limpieza con las siguientes claves:
        - 'date_columns': lista de columnas de fecha
        - 'numeric_columns': lista de columnas numéricas
        - 'text_columns': lista de columnas de texto
        - 'drop_duplicates': bool, si eliminar duplicados
        - 'handle_missing': dict con estrategias por columna
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame limpio
    dict
        Reporte de limpieza con estadísticas
    """
    
    # ESCRIBE TU CÓDIGO AQUÍ
    # 1. Crear una copia del dataframe
    # 2. Limpiar nombres de columnas
    # 3. Procesar columnas de fecha
    # 4. Procesar columnas numéricas
    # 5. Procesar columnas de texto
    # 6. Eliminar duplicados si está configurado
    # 7. Manejar valores perdidos según configuración
    # 8. Generar reporte de limpieza
    # 9. Retornar dataframe limpio y reporte
    
    pass  # Reemplaza con tu código

# Prueba tu función con los datos de empleados del Ejercicio 1
# df_clean, report = clean_dataframe(df_empleados, config={...})
# print(report)

---

## 🎓 Conclusión

La limpieza de datos no es glamurosa, pero es absolutamente esencial. Recuerda:

1. **Siempre preserva los datos originales** - Trabaja en copias
2. **Documenta tus decisiones** - El "por qué" es tan importante como el "qué"
3. **Valida constantemente** - No confíes, verifica
4. **Investiga antes de eliminar** - Los valores atípicos pueden ser señales, no ruido
5. **Automatiza cuando sea posible** - Crea funciones reutilizables
6. **La limpieza nunca es perfecta** - Saber cuándo parar es una habilidad

**La próxima vez que veas un modelo con 99% de precisión, pregunta primero sobre los datos, no sobre el modelo.**

Los datos limpios son la base de cualquier proyecto exitoso de ciencia de datos. Sin ellos, estás construyendo sobre arena.

---

### 📚 Recursos Adicionales

- **Documentación de Pandas**: https://pandas.pydata.org/docs/
- **PyJanitor**: https://pyjanitor-devs.github.io/pyjanitor/
- **Great Expectations**: https://greatexpectations.io/
- **ydata-profiling**: https://github.com/ydataai/ydata-profiling

### 🔑 Conceptos Clave

- **Valores perdidos**: Missing data, NaN, NULL
- **Duplicados**: Registros idénticos o casi idénticos
- **Valores atípicos**: Outliers, datos anómalos
- **Tipos de datos**: Data types, conversión de tipos
- **Normalización**: Estandarización de formatos
- **Validación**: Comprobación de integridad de datos
- **Pipeline de limpieza**: Flujo automatizado de limpieza

¡Buena suerte con tus proyectos de limpieza de datos! 🚀